# NFL Hierarchical Bayesian Model

Fits a hierarchical Bayesian model to NFL game data to estimate team-level
attack and defense strengths. Uses these estimates to simulate future games
and generate spread predictions.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az

from src.data import load_game_data
from src.model import bhm, simulate_team_seasons, predictions, plot_hdis

## Load Data

In [ ]:
season = 2024

# Load regular season games (first 14 weeks as training data)
train_weeks = list(range(1, 15))
df_train, teams = load_game_data(season, weeks=train_weeks)

print(f"Training on {len(df_train)} games from weeks {min(train_weeks)}-{max(train_weeks)}")
print(f"{len(teams)} teams")
df_train.head()

## Fit Model

In [ ]:
# Fit hierarchical Bayesian model
idata = bhm(df_train, metric='score', K=1, samples=1000)

# Check convergence
az.plot_trace(idata, var_names=['home', 'intercept', 'sd_att', 'sd_def'])
plt.tight_layout()
plt.show()

## Team Strength Estimates

In [ ]:
# Extract posterior means for attack and defense
atts_mean = idata.posterior['atts'].mean(dim=['chain', 'draw']).values
defs_mean = idata.posterior['defs'].mean(dim=['chain', 'draw']).values

team_strengths = teams.copy()
team_strengths['attack'] = atts_mean
team_strengths['defense'] = defs_mean
team_strengths = team_strengths.sort_values('attack', ascending=False)

print("Team Strength Estimates (higher attack = stronger offense, lower defense = stronger defense):")
print(team_strengths[['team', 'attack', 'defense']].to_string(index=False))

## Simulate and Predict

In [ ]:
# Load test weeks
test_weeks = list(range(15, 19))
df_test, _ = load_game_data(season, weeks=test_weeks)

# Simulate games
nsims = 1000
simuls = simulate_team_seasons(df_test, idata, nsims=nsims, metric='score')

# Generate predictions with 90% credible intervals
hdis = predictions(df_test, simuls, teams, nsims=nsims)

# Plot
fig, ax = plot_hdis(hdis, metric='score')
plt.show()

## Spread Predictions

In [ ]:
from src.model import simulate_team_season

# For each game, simulate outcomes and compute spread / win probability
game_preds = []

for _, game in df_test.iterrows():
    home = game['home_team']
    away = game['away_team']
    week = game['week']
    
    # Simulate this game many times
    home_scores_sim = []
    away_scores_sim = []
    for _ in range(nsims):
        sim = simulate_team_season(df_test[df_test.index == game.name], idata, metric='score', burnin=0)
        home_scores_sim.append(sim['home_score'].values[0])
        away_scores_sim.append(sim['away_score'].values[0])
    
    home_scores_sim = np.array(home_scores_sim)
    away_scores_sim = np.array(away_scores_sim)
    
    spread = np.median(away_scores_sim - home_scores_sim)
    home_win_prob = np.mean(home_scores_sim > away_scores_sim)
    
    actual_spread = game['away_score'] - game['home_score']
    home_won = game['home_score'] > game['away_score']
    
    game_preds.append({
        'week': week, 'home': home, 'away': away,
        'pred_spread': spread, 'home_win_prob': home_win_prob,
        'actual_spread': actual_spread, 'home_won': home_won,
        'correct': (home_win_prob > 0.5) == home_won,
    })

preds = pd.DataFrame(game_preds)
print(f"Accuracy: {preds['correct'].mean():.1%} ({preds['correct'].sum()}/{len(preds)})")
preds